[1] Dynamics of offshore structures, J. F. Wilson (Chapter 9.1)

In [1]:
import numpy as np

In [2]:
from sympy import symbols, Matrix, solve, sqrt, Array

In [3]:
from scipy.optimize import fsolve

In [4]:
from scipy.integrate import quad

### Platform parameters 

In [5]:
m1 = 4.69e6 # [kg]
m2 = 3.13e6 # [kg]

In [6]:
k11 = 7.35e7 # [N/m]
k22 = 3.59e8  # [N/m]
k12 = k21 = -1.15e8  # [N/m]

In [7]:
l1 = l2 = 38  # [m]
d = 61  # [m]

In [8]:
zeta1 = zeta2 = 0.05

In [9]:
Dl = 5.5 # [m]
Dc = 4.3 # [m]
Nl = 4
Nc = 2
w = 30 # [m]

### Wave loading parameters

In [10]:
H = 11.6 # [m]

In [11]:
T=15.4 # [sec]

In [12]:
k=0.0201 # [m^-1]

In [13]:
lambda_=312 # [m]

In [14]:
omega=0.408 # [rad/sec]

In [15]:
rho=1031 # [kg/m^3]

In [16]:
CM=2

In [17]:
g=9.81 # [m/sec^2]

### Calculations

In [18]:
K = Matrix([[k11,k12],
            [k21,k22]])

In [19]:
M = Matrix([[m1,0],
            [0,m2]])

In [20]:
omega_squad = symbols('omega_squad')

In [21]:
solutions = solve((K-omega_squad *M).det(), omega_squad )

In [22]:
solutions

[7.28428915294557, 123.083838261102]

In [23]:
omega1=sqrt(solutions[0])
omega2=sqrt(solutions[1])

In [24]:
omega1, omega2

(2.69894222853057, 11.0943155832662)

In [25]:
xi = symbols('xi')

In [26]:
solutions = (K-omega_squad *M)*Matrix([1,xi])

In [27]:
solutions[0]

-4690000.0*omega_squad - 115000000.0*xi + 73500000.0

In [28]:
solutions[1]

xi*(359000000.0 - 3130000.0*omega_squad) - 115000000.0

In [29]:
xi21_ = solve(solutions[0], xi)[0].subs(omega_squad, omega1**2)

In [30]:
xi22_ = solve(solutions[1], xi)[0].subs(omega_squad, omega2**2)

In [31]:
xi21_, xi22_

(0.342058120632046, -4.38054957777887)

In [32]:
xi_n=Matrix([[1,1],[xi21_,xi22_]])

In [33]:
e_squad=xi_n.transpose()*M*xi_n

In [34]:
e=e_squad**0.5

In [35]:
e_=Matrix([e[0,0], e[1,1]])

In [36]:
x1=xi_n.col(0)/e_[0]

In [37]:
x1

Matrix([
[0.000444720288978526],
[0.000152120186254935]])

In [38]:
x2=xi_n.col(1)/e_[1]

In [39]:
x2

Matrix([
[ 0.000124271802012958],
[-0.000544378789837681]])

In [40]:
X=x1.hstack(x1,x2)

In [41]:
X

Matrix([
[0.000444720288978526,  0.000124271802012958],
[0.000152120186254935, -0.000544378789837681]])

In [42]:
d/T**2*3.28

0.8436498566368694

In [43]:
H/T**2*3.28

0.16043177601619157

In [44]:
# hence Stoke's 2nd order

In [45]:
# intermediate water depth

In [46]:
def wavelength(_lambda_):
    return _lambda_-T*np.sqrt(g*_lambda_/(2*np.pi)*np.tanh(2*np.pi*d/_lambda_))

In [47]:
fsolve( wavelength, 100)

array([311.87576434])

In [48]:
_k_=2*np.pi/fsolve( wavelength, 100)[0]

In [49]:
def du(z,t,omega):
    return (-2*np.pi**2*H/T**2*np.cosh(k*(z+d))/np.sinh(k*d)*np.sin(omega*t)
            -3*np.pi**3*H**2/(T**2*lambda_)*np.cosh(2*k*(z+d))/np.sinh(k*d)**4*np.sin(2*omega*t))
    

In [50]:
def q_(z,t,N,D,omega):
    return N*CM*np.pi/4*rho*D**2*du(z,t,omega)

In [51]:
def y_(tau, x, p1, p2, omega_d, omega, t, zeta):
    return np.dot(x.T,np.array([p1,p2]))[0]/omega_d*np.exp(- zeta*omega*(t-tau))*np.sin(omega_d*(t-tau))

In [52]:
max1 = max2 = -float('inf')
for t in range(int(30/0.2)):
    t_ = t*0.2
    p1, err1 = quad(q_, 0, -(d-l2),args=(t_, Nl, Dl, float(omega1)))
    p21, err21 = quad(q_, -d, -(d-l2),args=(t_, Nl, Dl,float(omega2)))
    p2 = p21 + w*q_(-(d-l2), 0.2, Nc, Dc, float(omega2))
    y1, err_y1= quad(y_, 0, t_, args=(x1, p1, p2, 2.7026, 2.7060, t_, zeta1))
    y2, err_y2= quad(y_, 0, t_, args=(x2, p1, p2, 11.076, 11.090, t_, zeta2))
    xi1 = np.dot(x1.T,np.array([y1,y2]))[0]
    xi2 = np.dot(x2.T,np.array([y1,y2]))[0]
    max1 = max(max1,xi1)
    max2 = max(max2,xi2)

In [53]:
max1, max2

(0.150898605601624, 0.0502031941655377)

In [54]:
xi1_max=abs(max1)

In [55]:
xi2_max=abs(max2)

In [56]:
xi1_max, xi2_max

(0.150898605601624, 0.0502031941655377)